 ## Step 1: UNO Deck Construction
In this step, I am setting up the fundamental building blocks of the game according to the simplified rules:
* **Card Class:** Implements a `(color, value)` structure.
* **Deck Generator:** Creates a deck with 4 colors (Red, Blue, Green, Yellow) and values 0–9 plus "Skip" cards. 
* **Rule Compliance:** No Wild cards or Draw Four cards are included per the project requirements.

This serves as the "Initial State" of the deck before shuffling.

In [2]:
import random

# 1. Card Class required
class Card:
    def __init__(self, color, value):
        self.color = color
        self.value = value

    def __repr__(self):
        return f"{self.color} {self.value}"

# 2. Deck Generator
def create_deck():
    colors = ["Red", "Blue", "Green", "Yellow"]
    values = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "Skip"]
    deck = []
    
    for color in colors:
        for value in values:
            # Adding one card of each type per color for the simplified version
            deck.append(Card(color, value))
            # Usually UNO has two of each (except 0), let's add a second for variety
            if value != "0":
                deck.append(Card(color, value))
    
    return deck

# Initialize and test
full_deck = create_deck()
print(f"Deck created with {len(full_deck)} cards.")
print("First 5 cards:", full_deck[:5])

Deck created with 84 cards.
First 5 cards: [Red 0, Red 1, Red 1, Red 2, Red 2]


## Step 2: Shuffling and Dealing
This step handles the initialization of the game state:
* **Shuffling:** The deck is randomized using `random.shuffle`.
* **Dealing:** Each of the 3 players receives exactly 5 cards.
* **Discard Pile:** A starting "Top Card" is drawn from the deck.
* **Constraint Check:** Logic is added to ensure the starting card is a number card, not a "Skip" card.

In [3]:
import random

def setup_game(deck):
    # 1. Shuffle the deck
    random.shuffle(deck)
    
    # 2. Deal 5 cards to each of the 3 players
    # Using a dictionary to store hands for easy access
    hands = {
        "Player 1 (Minimax)": [],
        "Player 2 (Expectimax)": [],
        "Player 3 (User/AI)": []
    }
    
    for _ in range(5):
        for player in hands:
            hands[player].append(deck.pop())
            
    # 3. Set the initial Top Card
    # Ensure the first card isn't a "Skip" for a clean start
    top_card = deck.pop()
    while top_card.value == "Skip":
        deck.append(top_card)
        random.shuffle(deck)
        top_card = deck.pop()
        
    return hands, top_card, deck

# Execute the setup
player_hands, current_top_card, remaining_deck = setup_game(full_deck)

# Verification
print(f"Top Card to start: {current_top_card}")
for player, hand in player_hands.items():
    print(f"{player} hand: {hand}")
print(f"Cards left in deck: {len(remaining_deck)}")

Top Card to start: Green 2
Player 1 (Minimax) hand: [Yellow 4, Green 3, Green Skip, Yellow 6, Red 5]
Player 2 (Expectimax) hand: [Yellow Skip, Red 3, Green 9, Yellow 9, Yellow 5]
Player 3 (User/AI) hand: [Blue 2, Blue 8, Green 4, Blue 3, Yellow 3]
Cards left in deck: 68


## Step 3: Game State Visualization
To monitor the game flow and debug the AI search algorithms later, I implemented a display function:
* **Status Monitoring:** Shows the current top card and the number of cards left in the draw pile.
* **Hand Tracking:** Lists the cards currently held by Player 1 (Minimax), Player 2 (Expectimax), and Player 3.
* **Foundation for GUI:** This text-based output represents the data structure that will eventually feed into the GUI simulation.

In [5]:
def display_game_state(player_hands, top_card, deck):
    print("-" * 30)
    print(f" TOP CARD: [{top_card}] ")
    print(f" Deck: {len(deck)} cards remaining")
    print("-" * 30)
    
    for player, hand in player_hands.items():
        # Formatting the hand to look nicer
        hand_str = " | ".join([str(card) for card in hand])
        print(f"{player}:")
        print(f"  Hand: [{hand_str}]")
        print(f"  Count: {len(hand)} cards")
        print("-" * 15)

# Test the display
display_game_state(player_hands, current_top_card, remaining_deck)

------------------------------
 TOP CARD: [Green 2] 
 Deck: 68 cards remaining
------------------------------
Player 1 (Minimax):
  Hand: [Yellow 4 | Green 3 | Green Skip | Yellow 6 | Red 5]
  Count: 5 cards
---------------
Player 2 (Expectimax):
  Hand: [Yellow Skip | Red 3 | Green 9 | Yellow 9 | Yellow 5]
  Count: 5 cards
---------------
Player 3 (User/AI):
  Hand: [Blue 2 | Blue 8 | Green 4 | Blue 3 | Yellow 3]
  Count: 5 cards
---------------


## Step 4: Rule Implementation (Valid Moves)
I have implemented the core logic for "Rule 1" of the assignment:
* **Match Logic:** A card is valid if its color matches the top card OR its value matches the top card.
* **Skip Cards:** These are treated as valid moves if they match the current color.
* **Legal Move Generator:** This function will be used by the AI search trees to determine possible future states.

In [6]:
def get_valid_moves(hand, top_card):
    """
    Rule 1: A player may play a card if it has the same color OR same value.
    'Skip' cards follow the same color rule.
    """
    valid_moves = []
    
    for card in hand:
        if card.color == top_card.color or card.value == top_card.value:
            valid_moves.append(card)
            
    return valid_moves

# Quick Test
test_hand = player_hands["Player 1 (Minimax)"]
moves = get_valid_moves(test_hand, current_top_card)

print(f"Top Card is: {current_top_card}")
print(f"Hand is: {test_hand}")
print(f"Valid moves you can play: {moves}")

Top Card is: Green 2
Hand is: [Yellow 4, Green 3, Green Skip, Yellow 6, Red 5]
Valid moves you can play: [Green 3, Green Skip]


## Step 5: Turn-Based System and Skip Logic
This step implements the flow of the game and Rule 3:
* **Turn Management:** Uses a modular index to cycle through the 3 players.
* **Rule 2 Implementation:** Added logic for players to draw a card if no valid moves are available.
* **Rule 3 Implementation:** If a "Skip" card is played, the turn index increments by 2, bypassing the next player.
* **State Transition:** The function updates the top card and the player's hand, reflecting a change in the game state.

In [7]:
def play_turn(current_player_index, player_names, hands, top_card, deck):
    player_name = player_names[current_player_index]
    hand = hands[player_name]
    
    print(f"\n--- {player_name}'s Turn ---")
    valid_moves = get_valid_moves(hand, top_card)
    
    skip_occured = False
    
    if not valid_moves:
        # Rule 2: Drawing Card
        if len(deck) > 0:
            new_card = deck.pop()
            hand.append(new_card)
            print(f"{player_name} had no valid moves and drew: {new_card}")
        else:
            print(f"Deck is empty! {player_name} passes.")
    else:
        # For now, we just pick the first valid card (AI logic comes in Step 6 & 7)
        chosen_card = valid_moves[0]
        hand.remove(chosen_card)
        top_card = chosen_card
        print(f"{player_name} played: {chosen_card}")
        
        # Rule 3: Skip Card Logic
        if chosen_card.value == "Skip":
            skip_occured = True
            print("Next player's turn is SKIPPED!")

    # Calculate next turn
    # If skip occurred, move 2 spaces, otherwise move 1
    next_index = (current_player_index + (2 if skip_occured else 1)) % len(player_names)
    
    return next_index, top_card

# Testing a single cycle
player_list = list(player_hands.keys())
current_turn = 0

# Simulate one turn
current_turn, current_top_card = play_turn(current_turn, player_list, player_hands, current_top_card, remaining_deck)


--- Player 1 (Minimax)'s Turn ---
Player 1 (Minimax) played: Green 3
